# 2. FranceCrops references and full-year Sentinel-2 archive

Collect all eligible declared cabbage (`CHU`) and table-potato (`PTC`) parcels. FranceCrops has no carrot class. All upstream splits can be archived; their roles are retained, and reserved validation/test parcels are excluded from model-development exports.

The archive contains 24 calendar half-month medians per tile plus dated vegetation summaries. No timing gate, candidate budget or pilot pixel cap is applied here.

**Progress and time estimates:** the top bar counts completed stages; inner bars count finished patches, parcels, files, pixels or models. Stage ETA uses observed speed. Whole-notebook ETA uses a comparable completed run, or measured remaining work in the final stage; otherwise it says `learning`. Estimates are rough and include network variability. Long operations advance when their work unit finishes. The final cell prints elapsed time and the saved timing-report path.


## 1. Fresh-clone setup

Choose a Python 3.12+ kernel (3.13 recommended), then **Run All**. Missing packages and local settings are prepared automatically. Enter your Earth Engine project ID once if prompted; existing settings are reused. Collection and sign-in are enabled.

**Output:** the source-table and local archive paths. No pre-existing FranceCrops data is required: the next stage fetches missing reference tables.


In [ ]:
# Standard-library setup runs before scientific imports, so a fresh clone needs no saved data.
from pathlib import Path
import sys

notebook_repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebook_source_folder = str(notebook_repository_root / "src")
if notebook_source_folder not in sys.path:
    sys.path.insert(0, notebook_source_folder)
from crop_pipeline import notebook_setup
notebook_setup.prepare_environment(notebook_repository_root)

from pathlib import Path
import json
import os
from crop_pipeline.sources import francecrops as francecrops_workflow
from crop_pipeline.windows.seasons import file_sha256

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
revision = francecrops_workflow.DEFAULT_REVISION
source_table_folder = report_folder.parent / "francecrops/source_metadata" / revision
from crop_pipeline.sources import annual

run_name = annual.COLLECTION_NAME
# Run All starts collection; existing verified files are reused in this versioned archive.
download_missing_reference_tables = True
run_satellite_collection = True
authenticate_now = True
earth_engine_project = os.environ.get("EE_PROJECT", local_settings.get("earth_engine_project", ""))
annual_archive_root = annual.archive_root(local_settings, report_folder)
print("Collection:", run_name)
print("Archive root:", annual_archive_root)
print("Satellite collection enabled:", run_satellite_collection)


# Resolve account setup before large reference transfers start.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)

from crop_pipeline import progress as pipeline_progress
notebook_progress = pipeline_progress.start_notebook(
    "02_francecrops_preparation",
    ['Obtain reference tables', 'Prepare eligible parcels', 'Certificates', 'Collect annual imagery'],
    report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "satellite_collection": globals().get("run_satellite_collection"), "training_mode": globals().get("training_mode")},
)


## 2. Obtain the reference tables automatically

Fetch only missing class-map, label and footprint Parquet tables from the pinned FranceCrops release. Reuse existing files without overwriting them. Original interpolated image arrays are not needed. Certificate setup runs before these requests.

**Output:** three local table paths. For an offline inspection, set `download_missing_reference_tables = False`; missing inputs then produce a clear error.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Obtain reference tables')

table_paths = francecrops_workflow.download_reference_tables(
    source_table_folder, revision=revision,
    download_if_missing=download_missing_reference_tables,
)
print("Verified reference paths:", table_paths)

pipeline_progress.end_stage("completed")


## 3. Prepare all eligible parcels and estimate storage

Verify explicit class codes, WGS84 geometry, unique identities and metropolitan-France coordinates. Exclude competing parcel claims and apply the existing 10 m inward buffer. Retain every remaining pixel, using bounded tiles for large fields.

**Output:** `estimate` counts parcels, tile files and uncompressed raster GiB. The geometry audit records invalid footprints and the unavailable carrot class. Storage is estimated from the actual grids, before any image query.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Prepare eligible parcels')

reference_features, reference_audit = francecrops_workflow.load_target_parcels(table_paths, revision, return_audit=True)
annual_plan = annual.prepare_source(
    "francecrops", reference_features, annual_archive_root,
    provenance={"dataset_revision": revision, "tables": {name: file_sha256(path) for name, path in table_paths.items()},
                "unavailable_target_classes": ["carrot"], "upstream_splits_preserved": True, "reference_audit": reference_audit},
    inward_buffer_metres=10,
)
print("Estimate:", annual_plan["estimate"])
print("Pixels by year:", {year: inputs["pixels_per_crop"] for year, inputs in annual_plan["years"].items()})

pipeline_progress.set_workload({"folder": str(annual_archive_root), "source_revision": annual_plan["revision"], "satellite_collection": run_satellite_collection})

pipeline_progress.end_stage("completed")


## Collect the full calendar year — enabled for Run All

`run_satellite_collection = True` and `authenticate_now = True` are set for the next Run All. To prepare only, set both to `False`. Earth Engine applies the existing cloud masks, bilinear B5/B11 resampling, daily mosaics and half-month medians. It also calculates dated NDVI, NDRE and NDMI summaries. GeoTIFFs and summary tables are saved locally, one tile at a time, with verified-file reuse on restart.

**Output: `status: disabled`** means no authentication, image queries, tasks or downloads ran. **`completed`** means every planned tile has all 24 saved intervals and daily-summary checkpoints. A cloudy interval can be saved successfully and still fail the window notebook's quality checks.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Certificates')

import ssl
import certifi

# Use the system's explicit trust setting when one exists; otherwise use certifi.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
print("Trusted certificates:", ssl.create_default_context().cert_store_stats()["x509_ca"])

pipeline_progress.end_stage("completed")


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Collect annual imagery')

collection_result = annual.collect_source(
    annual_plan, report_folder, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
)
print(collection_result)

pipeline_progress.end_stage("completed" if run_satellite_collection else "skipped")
pipeline_progress.finish_notebook()


## Continue to notebook 04

Finish all three source notebooks before opening [04_four_month_windows.ipynb](04_four_month_windows.ipynb). It requires all three completed handoffs, applies the existing timing and quality rules, and selects the earliest window with complete usable pixels per parcel. It retains all usable pixels; the old 500-pixel pilot cap does not apply.

Changing timing thresholds does not require another satellite download while the required dates are already archived. The archive covers January–December of each reference year; a cycle crossing that boundary remains excluded until adjacent-year imagery is explicitly collected. Existing pilot files and results stay in their original folders.
